# Week 1 · Tuesday: Decision Trees and Clustering

Yesterday's models drew one line or one curve. Today:

1. **Decision trees:** a model that classifies by asking yes/no questions
2. **Unsupervised learning:** finding groups when there are no labels
3. **Distance:** between points, and between clusters
4. **K-means by hand,** one step at a time
5. **K-means with scikit-learn,** and how to choose K
6. **A real dataset:** irises
7. **Where K-means fails,** and DBSCAN

Run the next cell first. It holds the plotting helpers we reuse all day, so we can spend class time on the models instead of on matplotlib.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def plot_points(X, labels=None, title="", xlabel="Feature 1", ylabel="Feature 2",
                names=None, centers=None, ax=None):
    """Scatter 2D points, coloured by label if given. Label -1 is drawn as noise (DBSCAN)."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], s=45, edgecolor="black", alpha=0.85)
    else:
        for value in np.unique(labels):
            mask = labels == value
            if value == -1:
                ax.scatter(X[mask, 0], X[mask, 1], s=70, marker="x", color="black", label="Noise (-1)")
            else:
                label = names[value] if names is not None else f"{value}"
                ax.scatter(X[mask, 0], X[mask, 1], s=45, edgecolor="black", alpha=0.85, label=label)
        ax.legend()
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], s=300, marker="X", color="red",
                   edgecolor="black", linewidth=2, label="Centroids")
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if show:
        plt.show()


def plot_decision_boundary(model, X, y, title="", xlabel="Feature 1", ylabel="Feature 2",
                           names=None, ax=None):
    """Colour the background by what the model predicts at every point of a fine grid."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    grid_predictions = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    classes = np.unique(y)
    colours = ListedColormap(plt.cm.tab10.colors[:len(classes)])   # same colours as the points
    ax.contourf(xx, yy, grid_predictions, levels=np.arange(len(classes) + 1) - 0.5, alpha=0.25, cmap=colours)
    plot_points(X, y, title, xlabel, ylabel, names=names, ax=ax)
    if show:
        plt.show()


def plot_line_chart(xs, ys, title, xlabel, ylabel):
    """A simple line chart with markers, for the elbow and silhouette plots."""
    plt.figure(figsize=(8, 5))
    plt.plot(xs, ys, marker="o", linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(xs)
    plt.grid(alpha=0.3)
    plt.show()


print("Setup complete")

## 1. Decision trees

A decision tree classifies by asking a series of yes/no questions about the features, like a game of 20 questions:

```
Temperature <= 0.37?
   yes -> Humidity <= 1.2?  ...
   no  -> Play outside
```

scikit-learn learns which questions to ask, and in what order, from the training data.

### 1.1 Should we play outside?

A made-up weather app with two features and one yes/no label. `make_classification` builds data like this for practice.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

X_weather, y_weather = make_classification(
    n_samples=250, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.0, random_state=0
)
weather_names = ["Do not play", "Play"]
weather_features = ["Temperature-like", "Humidity-like"]

plot_points(X_weather, y_weather, "Synthetic weather data", *weather_features, names=weather_names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_weather, y_weather, test_size=0.2, random_state=RANDOM_STATE, stratify=y_weather
)

weather_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
weather_tree.fit(X_train, y_train)

print(f"Training accuracy: {accuracy_score(y_train, weather_tree.predict(X_train)):.3f}")
print(f"Test accuracy:     {accuracy_score(y_test, weather_tree.predict(X_test)):.3f}")

plot_decision_boundary(weather_tree, X_weather, y_weather, "Decision tree boundary",
                       *weather_features, names=weather_names)

- `max_depth` is a **hyperparameter**: the most questions the tree may ask in a row.
- `stratify=y_weather` keeps the same mix of classes in the train and test sets.
- The boundary is made of **straight, axis-aligned edges**. Each question looks at one feature at a time ("temperature below X?"), so every split is a horizontal or vertical cut.

The best part of a tree is that you can read it:

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(weather_tree, feature_names=weather_features, class_names=weather_names,
          filled=True, rounded=True, fontsize=10)
plt.title("The questions the weather tree asks")
plt.show()

for name, importance in zip(weather_features, weather_tree.feature_importances_):
    print(f"{name:<18} importance {importance:.3f}")

Each box shows the question, how many training samples reached it (`samples`), the class split (`value`), and **gini**, a measure of how mixed the classes are (0 = all one class). The tree picks the question that makes the groups below it as pure as possible.

**Feature importance** sums how much each feature's questions helped. It's a quick way to explain *what the model is looking at*.

### 1.2 Harder shapes, and how deep is too deep

Two interlocking half-moons can't be split with one straight line. Let's compare a shallow tree, a medium one, and one with no depth limit, on training **and** test data.

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(X_moons, y_moons, test_size=0.25, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, depth in zip(axes, [1, 4, None]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    train_acc = accuracy_score(y_train, tree.predict(X_train))
    test_acc = accuracy_score(y_test, tree.predict(X_test))
    plot_decision_boundary(tree, X_train, y_train, ax=ax,
                           title=f"max_depth={depth}\ntrain {train_acc:.2f} | test {test_acc:.2f}")
plt.tight_layout()
plt.show()

- **Depth 1** underfits: one cut, so both scores are low.
- **Depth 4** captures the shape: both scores are good.
- **No limit** keeps splitting until every training point is classified correctly (training accuracy 1.00), carving tiny boxes around noisy points. Its test score drops: **overfitting**, the same trade-off as yesterday's polynomial degrees.

Limiting depth (or **pruning**) is how you keep a tree honest. Random forests, many trees voting together, are the other common fix.

### 1.3 Trees can do regression too

A `DecisionTreeRegressor` predicts a number instead of a class. Each leaf predicts the average of the training points that landed in it, so the prediction looks like **stair steps**.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

x = np.linspace(0, 10, 200)
y_wave = np.sin(x) + np.random.normal(0, 0.1, x.shape)    # something that cycles up and down

regression_tree = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)
regression_tree.fit(x.reshape(-1, 1), y_wave)

xs = np.linspace(0, 10, 500).reshape(-1, 1)
plt.figure(figsize=(9, 5))
plt.scatter(x, y_wave, edgecolor="black", alpha=0.6, label="Noisy data")
plt.plot(xs, regression_tree.predict(xs), color="red", linewidth=3, label="Tree prediction (depth 5)")
plt.title("Decision tree regression")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. Unsupervised learning

Everything so far was **supervised**: every sample came with the right answer. **Unsupervised** learning gets only the features, and looks for structure on its own.

The most common task is **clustering**: putting similar samples into groups.

- **Customer segmentation:** group customers by income and spending, then market to each group.
- **Document grouping:** cluster support tickets or articles by topic.
- **Anomaly detection:** points that fit no group are worth a look.
- **Image compression:** replace thousands of colours with a few cluster colours.

The catch: there's no "right answer" to check against, so we need other ways to judge whether the groups are any good.

## 3. Distance

Clustering rests on one idea: **similar points are close together**. So first, what does "close" mean?

### 3.1 Distance between two points

In [ ]:
a = np.array([1, 2])
b = np.array([4, 6])

euclidean = np.sqrt(np.sum((a - b) ** 2))     # straight line: the hypotenuse
manhattan = np.sum(np.abs(a - b))             # city blocks: across, then up

print(f"Euclidean distance: {euclidean}")
print(f"Manhattan distance: {manhattan}")
print(f"NumPy agrees:       {np.linalg.norm(a - b)}")

plt.figure(figsize=(5, 5))
plt.plot([a[0], b[0]], [a[1], b[1]], "r-", linewidth=3, label=f"Euclidean = {euclidean}")
plt.plot([a[0], b[0], b[0]], [a[1], a[1], b[1]], "b--", linewidth=3, label=f"Manhattan = {manhattan}")
plt.scatter(*zip(a, b), s=150, color="black", zorder=3)
plt.legend()
plt.grid(alpha=0.3)
plt.axis("equal")
plt.title("Two ways to measure distance")
plt.show()

- **Euclidean** (the default almost everywhere): the straight-line distance, `sqrt((x2-x1)² + (y2-y1)²)`.
- **Manhattan:** the distance walking a city grid, `|x2-x1| + |y2-y1|`.
- **Cosine similarity** (later in the course) compares directions instead of distances, and is how text embeddings are compared.

### 3.2 Distance between two clusters

Distance between two **groups** of points can be measured several ways. Each choice is called a **linkage**:

| Linkage | Distance between clusters A and B |
|---|---|
| **Single** | the two **closest** points, one from each |
| **Complete** | the two **farthest** points |
| **Average** | the average over every pair |
| **Centroid** | between the two cluster **centres** (means) |

In [ ]:
from scipy.spatial.distance import cdist

cluster_a = np.array([[1.0, 1.0], [1.5, 2.0], [2.0, 1.2]])
cluster_b = np.array([[5.0, 4.0], [6.0, 5.5], [5.5, 3.5]])

pairwise = cdist(cluster_a, cluster_b)      # every point in A against every point in B
print("Every A-to-B distance:")
print(np.round(pairwise, 2))
print()
print(f"Single   (closest pair):  {pairwise.min():.2f}")
print(f"Complete (farthest pair): {pairwise.max():.2f}")
print(f"Average  (all pairs):     {pairwise.mean():.2f}")
centroid_a, centroid_b = cluster_a.mean(axis=0), cluster_b.mean(axis=0)
print(f"Centroid (centre to centre): {np.linalg.norm(centroid_a - centroid_b):.2f}")

i, j = np.unravel_index(pairwise.argmin(), pairwise.shape)
k, l = np.unravel_index(pairwise.argmax(), pairwise.shape)
plt.figure(figsize=(7, 6))
plt.scatter(cluster_a[:, 0], cluster_a[:, 1], s=120, edgecolor="black", label="Cluster A")
plt.scatter(cluster_b[:, 0], cluster_b[:, 1], s=120, edgecolor="black", label="Cluster B")
plt.plot(*zip(cluster_a[i], cluster_b[j]), "g-", linewidth=2, label="Single (closest)")
plt.plot(*zip(cluster_a[k], cluster_b[l]), "r-", linewidth=2, label="Complete (farthest)")
plt.plot(*zip(centroid_a, centroid_b), "k--", linewidth=2, label="Centroid")
plt.scatter(*zip(centroid_a, centroid_b), marker="X", s=250, color="black")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Distance between clusters depends on the linkage")
plt.show()

### 3.3 Hierarchical clustering: linkage in action

**Hierarchical clustering** starts with every point as its own cluster, then repeatedly merges the two **closest** clusters, where "closest" is decided by the linkage. The merge history is a **dendrogram**: a family tree of clusters. Cut it at any height to get that many groups.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.datasets import make_blobs

X_small, _ = make_blobs(n_samples=15, centers=3, cluster_std=0.8, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_points(X_small, title="15 unlabelled points", ax=axes[0])
for index, point in enumerate(X_small):
    axes[0].annotate(str(index), point, textcoords="offset points", xytext=(6, 6))

dendrogram(linkage(X_small, method="average"), ax=axes[1])
axes[1].set_title("Dendrogram (average linkage)")
axes[1].set_xlabel("Point index")
axes[1].set_ylabel("Merge distance")
plt.tight_layout()
plt.show()

Points that merge low down are close neighbours. The tall vertical lines at the top are big jumps in distance: cutting the tree just below them gives **three clusters**, the natural grouping.

`method="single"`, `"complete"` and `"average"` are the linkages from 3.2. Try changing it: single linkage tends to chain points into long, stringy clusters, while complete linkage prefers compact ones.

## 4. K-means by hand

**K-means** is the most popular clustering algorithm. You choose **K**, the number of clusters, and it repeats two steps:

1. **Assign:** each point joins its nearest **centroid** (cluster centre).
2. **Update:** each centroid moves to the mean of the points assigned to it.

It stops when the centroids stop moving. Let's watch it.

In [ ]:
X_blobs, _ = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=7)

def assign_to_nearest(X, centroids):
    distances = cdist(X, centroids)          # each point against each centroid
    return distances.argmin(axis=1)          # index of the closest centroid

def move_centroids(X, labels, k):
    return np.array([X[labels == cluster].mean(axis=0) for cluster in range(k)])

k = 3
rng = np.random.default_rng(3)
centroids = X_blobs[rng.choice(len(X_blobs), size=k, replace=False)]   # start at 3 random points

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for step, ax in enumerate(axes.ravel(), start=1):
    labels = assign_to_nearest(X_blobs, centroids)
    plot_points(X_blobs, labels, title=f"Step {step}: assign, then move", centers=centroids, ax=ax)
    new_centroids = move_centroids(X_blobs, labels, k)
    print(f"Step {step}: centroids moved {np.linalg.norm(new_centroids - centroids):.3f}")
    centroids = new_centroids
plt.tight_layout()
plt.show()

Each panel shows the assignment with the centroids that produced it. The movement shrinks every step, and at **0** the algorithm has **converged**.

Two things to notice:

- The result depends on where the centroids **start**. A bad start can land in a worse grouping, so libraries run K-means several times from different starts and keep the best (`n_init`).
- **Nobody told it what K should be.** We did. Choosing K is the hard part.

## 5. K-means with scikit-learn

Imagine customers described by **annual income** and a **spending score**. We want the computer to find the customer segments on its own.

In [ ]:
from sklearn.cluster import KMeans

X_customers, _ = make_blobs(n_samples=500, centers=5, cluster_std=1.2, random_state=10)

plot_points(X_customers, title="Raw customer data: what groups do you see?",
            xlabel="Income", ylabel="Spending score")

In [ ]:
kmeans = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE)
kmeans.fit(X_customers)

print(f"Cluster of the first 10 customers: {kmeans.labels_[:10]}")
print(f"Inertia: {kmeans.inertia_:.1f}")

plot_points(X_customers, kmeans.labels_, "K-means with K = 5", "Income", "Spending score",
            centers=kmeans.cluster_centers_)

- `labels_`: the cluster each customer was put in. The numbers are just names; cluster 3 isn't "more" than cluster 1.
- `cluster_centers_`: the final centroids.
- **Inertia**: the sum of squared distances from every point to its centroid. Lower means tighter clusters.

### 5.1 Choosing K

We said K = 5 because we made the data. Normally we don't know. Two common guides, neither perfect:

**The elbow method.** Inertia always drops as K grows (with K = number of points it's 0). Look for the **elbow**, where adding another cluster stops helping much.

In [ ]:
k_values = list(range(1, 11))
inertias = [KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_customers).inertia_
            for k in k_values]

plot_line_chart(k_values, inertias, "Elbow method", "Number of clusters (K)", "Inertia")

**The silhouette score.** For each point, compare how close it is to its own cluster against how close it is to the next-nearest cluster. Averaged over all points it runs from **-1** (wrong clusters) through **0** (overlapping) to **1** (well separated). Higher is better. It needs at least 2 clusters.

In [ ]:
from sklearn.metrics import silhouette_score

k_values = list(range(2, 11))
scores = []
for k in k_values:
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_customers)
    scores.append(silhouette_score(X_customers, labels))

plot_line_chart(k_values, scores, "Silhouette score by K", "Number of clusters (K)", "Silhouette score")
print(f"Best silhouette: K = {k_values[int(np.argmax(scores))]}")

Both point at 5 here, because we built the data with five clear groups. Real data is rarely that tidy (section 6 shows a case where the metrics disagree with reality). Use them together, plus what you know about the problem. If marketing can only run four campaigns, K = 4 may be the right answer whatever the chart says.

### 5.2 Distance means scale matters

K-means measures Euclidean distance, so a feature with big numbers dominates. Here's the same kind of customer data in real units: income in **dollars**, spending score from **1 to 100**.

In [ ]:
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(1)
groups = [(40_000, 20), (40_000, 80), (90_000, 20), (90_000, 80)]   # four real segments
X_real = np.vstack([
    np.column_stack([rng.normal(income, 6_000, 60), rng.normal(score, 7, 60)])
    for income, score in groups
])

raw_labels = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit_predict(X_real)
X_scaled = StandardScaler().fit_transform(X_real)
scaled_labels = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_points(X_real, raw_labels, "K-means on raw units", "Income ($)", "Spending score", ax=axes[0])
plot_points(X_real, scaled_labels, "K-means after StandardScaler", "Income ($)", "Spending score", ax=axes[1])
plt.tight_layout()
plt.show()

On raw units, a difference of 10 spending points is nothing next to a difference of $10,000, so K-means effectively clusters on income alone and slices the data into vertical bands. After **scaling** each feature to the same range, both features count, and it finds the four real segments.

> Scale your features before any distance-based algorithm.

## 6. A real dataset: irises

The iris dataset measures 150 flowers from three species. We'll pretend we don't know the species, cluster on **petal length and width**, and only at the end peek at the real answers. (With real unsupervised data, you never get that peek.)

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data[:, [2, 3]]       # petal length, petal width

plot_points(X_iris, title="Iris petals (no labels)", xlabel="Petal length (cm)", ylabel="Petal width (cm)")

k_values = list(range(2, 9))
iris_scores = [silhouette_score(X_iris, KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_iris))
               for k in k_values]
plot_line_chart(k_values, iris_scores, "Iris: silhouette score by K", "Number of clusters (K)", "Silhouette score")

The silhouette score prefers **K = 2**. Look at the scatter: one species sits on its own, while the other two overlap. The metric is right about the *shape* of the data, and still doesn't match the three species we know exist. Metrics guide K; they don't decide it.

Let's use K = 3 and compare against the real species.

In [ ]:
iris_kmeans = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_points(X_iris, iris.target, "Actual species", "Petal length (cm)", "Petal width (cm)",
            names=list(iris.target_names), ax=axes[0])
plot_points(X_iris, iris_kmeans.labels_, "K-means clusters (K = 3)", "Petal length (cm)", "Petal width (cm)",
            centers=iris_kmeans.cluster_centers_, ax=axes[1])
plt.tight_layout()
plt.show()

Very close, with mistakes only where versicolor and virginica overlap. The cluster numbers won't match the species numbers, because K-means doesn't know the names; it just numbers its groups.

## 7. Where K-means fails, and DBSCAN

K-means assumes clusters are roughly round blobs around a centre. Give it two rings, one inside the other:

In [ ]:
from sklearn.datasets import make_circles
from sklearn.cluster import DBSCAN

X_circles, y_circles = make_circles(n_samples=500, noise=0.05, factor=0.45, random_state=RANDOM_STATE)

kmeans_circles = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit_predict(X_circles)
dbscan_circles = DBSCAN(eps=0.25, min_samples=5).fit_predict(X_circles)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_points(X_circles, kmeans_circles, "K-means (K = 2)", ax=axes[0])
plot_points(X_circles, dbscan_circles, "DBSCAN (eps = 0.25)", ax=axes[1])
plt.tight_layout()
plt.show()

print(f"DBSCAN found {len(set(dbscan_circles) - {-1})} clusters and {np.sum(dbscan_circles == -1)} noise points")

K-means cuts the rings in half, because "nearest centre" can only draw straight boundaries between centres.

**DBSCAN** (density-based clustering) works differently: points with enough close **neighbours** are grouped, and the group grows from neighbour to neighbour, so it can follow any shape.

- `eps`: how close counts as a neighbour.
- `min_samples`: how many neighbours make a point part of a dense region.
- You **don't** choose the number of clusters; it finds them.
- Points in no dense region get the label **-1**: noise. K-means can't flag outliers at all.

The trade-off: DBSCAN is slower on big data and sensitive to `eps`. Try `eps=0.1` and `eps=0.5` in the cell above.

## Recap

| Algorithm | Type | Good at | Watch out for |
|---|---|---|---|
| Decision tree | supervised | explainable rules, any shape | overfitting without `max_depth` |
| Hierarchical | unsupervised | seeing structure at every level (dendrogram) | slow on large data |
| K-means | unsupervised | fast, round clusters | you choose K; needs scaling; fails on odd shapes |
| DBSCAN | unsupervised | odd shapes, finds outliers | sensitive to `eps` |

**Quick check**
1. Why are a decision tree's boundaries made of straight horizontal and vertical edges?
2. A tree scores 1.00 on training data and 0.80 on test data. What happened, and what would you change?
3. What's the difference between single and complete linkage?
4. What are K-means' two repeated steps, and when does it stop?
5. Why should you scale features before K-means?
6. Name a dataset where DBSCAN would beat K-means.